# ==============================================================
# CELL 1 – Libraries, Canonical Column Schema, and Data Loader
# ==============================================================

In [4]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

# ── Canonical column order (must match aft_data.xlsx exactly) ──────────────
COLUMNS_ORDER = [
    "property_id",
    "description",
    "ad_number",
    "ad_date",
    "location",
    "price",
    "total_area",
    "net_area",
    "rooms_count",
    "halls_count",
    "bathrooms_count",
    "balconies_count",
    "floor_raw",
    "total_floors",
    "building_age",
    "legal_status",
    "house_condition",
    "heating_type",
    "kitchen_type",
    "has_elevator",
    "has_parking",
    "in_complex",
    "is_furnished",
    "facade_directions",
    "advertiser_type",
    "near_mosque",
    "near_hospital",
    "near_park",
    "near_university",
    "near_school",
    "near_market",
    "near_transport",
    "ad_url",
]

# Features used for content-based duplicate detection (besides ad_number)
DUPLICATE_FINGERPRINT_COLS = [
    "location", "price", "total_area",
    "rooms_count", "floor_raw", "total_floors",
]

FILE_PATH = "..\\data\\homs_properties_raw.xlsx"


def load_data(file_path: str = FILE_PATH) -> pd.DataFrame:
    """Load dataset from Excel; return empty frame with schema if file absent."""
    if not os.path.exists(file_path):
        print(f"⚠️  File not found — creating empty DataFrame with schema: {file_path}")
        return pd.DataFrame(columns=COLUMNS_ORDER)

    df = pd.read_excel(file_path)
    print(f"✅ Loaded {len(df):,} records successfully from → {file_path}")
    return df


def save_data(df: pd.DataFrame, file_path: str = FILE_PATH) -> None:
    """Save DataFrame back to Excel preserving canonical column order."""
    for col in COLUMNS_ORDER:
        if col not in df.columns:
            df[col] = np.nan
    df[COLUMNS_ORDER].to_excel(file_path, index=False)
    print(f"✅ Saved updated database ({len(df):,} records) → {file_path}")


def property_exists_by_ad_number(df: pd.DataFrame, ad_number) -> bool:
    """Return True if the advertisement number already exists in the database."""
    return str(ad_number) in df["ad_number"].astype(str).values


def property_exists_by_fingerprint(df: pd.DataFrame, raw: dict) -> bool:
    """
    Content-based duplicate check: return True if all 6 fingerprint columns
    match an existing row exactly (catches re-listings with a different ad_number).
    Comparison is done on string-normalized values to avoid type-mismatch misses.
    """
    if df.empty:
        return False

    fingerprint = {
        col: str(raw.get(col, "")).strip()
        for col in DUPLICATE_FINGERPRINT_COLS
    }

    mask = pd.Series([True] * len(df), index=df.index)
    for col, val in fingerprint.items():
        if col not in df.columns:
            continue
        mask &= df[col].astype(str).str.strip() == val

    return mask.any()


# ── Initial load ───────────────────────────────────────────────────────────
df_homs = load_data()


✅ Loaded 100 records successfully from → ..\data\homs_properties_raw.xlsx


# ==============================================================
# CELL 2 – Manual Input Dict (modify this cell for each new property)
# ==============================================================

In [5]:
new_property_input = {
    "description"      : "شقة مربحة للبيع في منطقة وادي الذهب بحجم تفرعات شارع التوتر جانب محصصة الياسمين، مساحة واسعة وتصميم جميل. الشقة بمساحة كلية 115 متر مربع وصافية 100 متر مربع، تتضمن 4 غرف وصالة، عمر البناء من 5 إلى 10 سنوات. تقع في الطابق الأول من بناء مكون من 3 طوابق، لا توجد مصاعد أو مواقف سيارات. تدفئة على الديزل لضمان الدفع في الشتاء، والمطبخ مغلق. شرفة واحدة تطل على الجهة الغربية. يوجد 2 حمام لخدمة الأسرة والضيوف. حالياً مسكونة من قبل صاحبها، الطابو كاتب عدل (وكالة غير قابلة للعزل). الشقة غير مفروشة، وتقع ضمن مجمع سكني. الموقع في منطقة مركزية يسهل الوصول إليها بالقرب من الخدمات في حمص. تواصل معنا عبر الواتساب",
    "ad_number"        : "010101004232098",
    "ad_date"          : "2026-05-21",
    "location"         : "وادي الذهب",
    "price"            : 23000,
    "total_area"       : "115",
    "net_area"         : "100",
    "rooms_count"      : "4",
    "halls_count"      : "1",
    "bathrooms_count"  : "2",
    "balconies_count"  : "1",
    "floor_raw"        : "1",
    "total_floors"     : "3",
    "building_age"     : "من 5 إلى 10 سنوات",
    "legal_status"     : "كاتب عدل (وكالة غير قابلة للعزل)",
    "house_condition"  : "مسكون من صاحبه",
    "heating_type"     : "مدفئة ديزل",
    "kitchen_type"     : "مغلق",
    "has_elevator"     : "لا يوجد",
    "has_parking"      : "لا يوجد",
    "in_complex"       : "نعم",
    "is_furnished"     : "غير مفروش",
    "facade_directions": "غرب, شرق",
    "advertiser_type"  : "وسيط",
    "ad_url"           : "https://doushesh.com/listing/shk-4-ghrf-llbyaa-fy-oady-althhb-bhms-115-m2-93639",
}


# ==============================================================
# CELL 3 – Automation Logic & Knowledge-Base Feature Extraction
# ==============================================================

In [6]:
def extract_proximity_features(description: str) -> dict:
    PROXIMITY_RULES = {
        "near_mosque"    : ["جامع", "مسجد"],
        "near_hospital"  : ["مشفى", "مستشفى", "صيدلية", "مركز طبي"],
        "near_park"      : ["الحديقة العامة", "منتزه", "متنزه"],
        "near_university": ["جامعة", "كلية"],
        "near_school"    : ["مدرسة", "مدرسه", "روضة"],
        "near_market"    : ["سوق", "أسواق", "مول", "بقالية", "محل تجاري"],
        "near_transport" : ["باص", "مواصلات", "كراج", "محطة باص", "موقف باصات"],
    }

    if pd.isna(description):
        return {col: 0 for col in PROXIMITY_RULES}

    # Single lower() allocation — reused for all keyword checks
    desc_lower = str(description).lower()

    return {
        flag: int(any(keyword in desc_lower for keyword in keywords))
        for flag, keywords in PROXIMITY_RULES.items()
    }


def build_property_row(raw: dict, df: pd.DataFrame) -> dict | None:
    ad_num = str(raw.get("ad_number", "")).strip()

    # Layer 1: ad_number uniqueness
    if property_exists_by_ad_number(df, ad_num):
        print(f"❌ Duplicate blocked — ad_number '{ad_num}' already exists in database.")
        return None

    # Layer 2: content fingerprint uniqueness
    if property_exists_by_fingerprint(df, raw):
        fp_vals = {col: raw.get(col) for col in DUPLICATE_FINGERPRINT_COLS}
        print(
            f"❌ Duplicate blocked — a property with identical core attributes already exists:\n"
            f"   {fp_vals}"
        )
        return None

    # Auto-increment property_id
    if not df.empty and "property_id" in df.columns and df["property_id"].notna().any():
        new_id = int(df["property_id"].max()) + 1
    else:
        new_id = 1

    # Build structured row from canonical schema; missing keys default to NaN
    row = {col: raw.get(col, np.nan) for col in COLUMNS_ORDER}

    # Apply automated enrichments
    row["property_id"] = new_id
    row.update(extract_proximity_features(raw.get("description", "")))

    return row


# ── Execute ────────────────────────────────────────────────────────────────
new_row = build_property_row(new_property_input, df_homs)
if new_row:
    print(f"✅ Logic applied successfully for Property ID: {new_row['property_id']}")


✅ Logic applied successfully for Property ID: 124


# ==============================================================
# CELL 4 – Commit to File and Visual Verification
# ==============================================================

In [7]:
if new_row is not None:
    new_df  = pd.DataFrame([new_row])
    df_homs = pd.concat([df_homs, new_df], ignore_index=True)

    save_data(df_homs, FILE_PATH)

    print("\n📊 Verification — Last 3 rows in database:")
    display(df_homs.tail(3)[[
        "property_id", "location", "price",
        "near_school", "near_transport",
    ]])


✅ Saved updated database (101 records) → ..\data\homs_properties_raw.xlsx

📊 Verification — Last 3 rows in database:


,property_id,location,price,near_school,near_transport
98,109,المحطة,250000,0,0
99,110,الزهراء,300000,0,0
100,124,وادي الذهب,23000,0,0
